# External Validation Figures

Zero-shot and finetuned C-index comparison (SPARC-Risk vs Image Only) on:
- SurGen CRC (COAD, READ)
- NLST (mean slide aggregation)
- MGH GI (Upper GI + 24 CRC, n=94)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# ── Load results from clean notebooks ─────────────────────────────────────────
import json as _json

def load_cohort(path):
    with open(path) as f:
        return _json.load(f)

surgen = load_cohort('../results/external_results/surgen.json')
nlst = load_cohort('../results/external_results/nlst.json')
mgh = load_cohort('../results/external_results/mgh_gi.json')

# Build results dict
def extract(cohort_data, breakdown='All'):
    b = cohort_data['breakdowns'][breakdown]
    r = {'n': b['n'], 'ev_dss': b.get('ev_dss', 0), 'ev_os': b.get('ev_os', 0)}
    for mode in ['zs', 'ft']:
        for ep in ['dss', 'os']:
            r[f'{mode}_{ep}'] = (b.get(f'{mode}_{ep}_sq', 0), b.get(f'{mode}_{ep}_img', 0))
            r[f'{mode}_{ep}_sq_ci'] = (b.get(f'{mode}_{ep}_sq_lo', None), b.get(f'{mode}_{ep}_sq_hi', None))
            r[f'{mode}_{ep}_img_ci'] = (b.get(f'{mode}_{ep}_img_lo', None), b.get(f'{mode}_{ep}_img_hi', None))
    for ep in ['dss', 'os']:
        r[f'hr_{ep}'] = (b.get(f'hr_{ep}_sq'), b.get(f'hr_{ep}_img'))
    return r

results = {
    'SurGen COAD': extract(surgen, 'COAD'),
    'SurGen READ': extract(surgen, 'READ'),
    'NLST': extract(nlst, 'All'),
    'MGH GI': extract(mgh, 'All'),
}

COHORTS = ['SurGen COAD', 'SurGen READ', 'NLST', 'MGH GI']

# Colors
C_IMG = '#D1D1D1'
C_SQ  = '#2B6B8A'

# Print summary
for cohort in COHORTS:
    r = results[cohort]
    print(f'{cohort} (n={r["n"]}):')
    for ep in ['dss', 'os']:
        for mode in ['zs', 'ft']:
            vals = r[f'{mode}_{ep}']
            ci_sq = r.get(f'{mode}_{ep}_sq_ci', (None, None))
            ci_img = r.get(f'{mode}_{ep}_img_ci', (None, None))
            ci_str = ''
            if ci_sq[0] is not None:
                ci_str = f' [{ci_sq[0]:.2f},{ci_sq[1]:.2f}] vs [{ci_img[0]:.2f},{ci_img[1]:.2f}]'
            d = vals[0] - vals[1]
            print(f'  {ep.upper()} {mode.upper()}: {vals[0]:.3f} vs {vals[1]:.3f} ({d:+.3f}){ci_str}')

In [ ]:
# ── Zero-Shot C-index ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(14, 4.5), sharey=True)

w = 0.35
endpoints = ['DSS', 'OS']

for ax, cohort in zip(axes, COHORTS):
    r = results[cohort]
    x = np.arange(len(endpoints))

    sq_vals = [r['zs_dss'][0], r['zs_os'][0]]
    img_vals = [r['zs_dss'][1], r['zs_os'][1]]

    # Bootstrap CIs (if available)
    sq_errs = [None, None]
    img_errs = [None, None]
    for i, ep in enumerate(['dss', 'os']):
        ci_sq = r.get(f'zs_{ep}_sq_ci', (None, None))
        ci_img = r.get(f'zs_{ep}_img_ci', (None, None))
        if ci_sq[0] is not None:
            sq_errs[i] = (sq_vals[i] - ci_sq[0], ci_sq[1] - sq_vals[i])
            img_errs[i] = (img_vals[i] - ci_img[0], ci_img[1] - img_vals[i])

    has_ci = sq_errs[0] is not None
    if has_ci:
        sq_yerr = [[e[0] for e in sq_errs], [e[1] for e in sq_errs]]
        img_yerr = [[e[0] for e in img_errs], [e[1] for e in img_errs]]
        ax.bar(x - w/2, img_vals, w, yerr=img_yerr, capsize=2,
               error_kw={'elinewidth': 0.8, 'ecolor': (0, 0, 0, 0.35)},
               color=C_IMG, edgecolor='black', linewidth=0.5,
               label='Image Only' if cohort == COHORTS[0] else '')
        ax.bar(x + w/2, sq_vals, w, yerr=sq_yerr, capsize=2,
               error_kw={'elinewidth': 0.8, 'ecolor': (0, 0, 0, 0.35)},
               color=C_SQ, edgecolor='black', linewidth=0.5,
               label='SPARC-Risk' if cohort == COHORTS[0] else '')
    else:
        ax.bar(x - w/2, img_vals, w, color=C_IMG, edgecolor='black', linewidth=0.5,
               label='Image Only' if cohort == COHORTS[0] else '')
        ax.bar(x + w/2, sq_vals, w, color=C_SQ, edgecolor='black', linewidth=0.5,
               label='SPARC-Risk' if cohort == COHORTS[0] else '')

    # Delta annotations (rounded to 2 decimals)
    for i, ep in enumerate(endpoints):
        d = sq_vals[i] - img_vals[i]
        y_top = max(sq_vals[i], img_vals[i])
        if has_ci:
            y_top += max(sq_errs[i][1], img_errs[i][1])
        y_top += 0.008
        color = '#2B6B8A' if d > 0 else '#999999'
        d_rounded = round(d, 2)
        ax.text(i, y_top, f'{d_rounded:+.2f}', ha='center', va='bottom', fontsize=8,
                color=color, fontweight='bold' if abs(d_rounded) >= 0.01 else 'normal')

    ax.set_xticks(x)
    ax.set_xticklabels(endpoints, fontsize=10)
    ax.set_title(f'{cohort}\n(n={r["n"]})', fontsize=11)
    ax.set_ylim(0.48, 0.85)
    ax.grid(axis='y', alpha=0.2)
    ax.axhline(y=0.5, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)

axes[0].set_ylabel('C-index', fontsize=12)
fig.legend(['Image Only', 'SPARC-Risk'], loc='upper right',
           bbox_to_anchor=(0.98, 0.98), fontsize=10)
fig.suptitle('Zero-Shot External Validation', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figs/external_zeroshot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Finetuned C-index ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(14, 4.5), sharey=True)

w = 0.35
endpoints = ['DSS', 'OS']

for ax, cohort in zip(axes, COHORTS):
    r = results[cohort]
    x = np.arange(len(endpoints))

    sq_vals = [r['ft_dss'][0], r['ft_os'][0]]
    img_vals = [r['ft_dss'][1], r['ft_os'][1]]

    # Bootstrap CIs (if available)
    sq_errs = [None, None]
    img_errs = [None, None]
    for i, ep in enumerate(['dss', 'os']):
        ci_sq = r.get(f'ft_{ep}_sq_ci', (None, None))
        ci_img = r.get(f'ft_{ep}_img_ci', (None, None))
        if ci_sq[0] is not None:
            sq_errs[i] = (sq_vals[i] - ci_sq[0], ci_sq[1] - sq_vals[i])
            img_errs[i] = (img_vals[i] - ci_img[0], ci_img[1] - img_vals[i])

    has_ci = sq_errs[0] is not None
    if has_ci:
        sq_yerr = [[e[0] for e in sq_errs], [e[1] for e in sq_errs]]
        img_yerr = [[e[0] for e in img_errs], [e[1] for e in img_errs]]
        ax.bar(x - w/2, img_vals, w, yerr=img_yerr, capsize=2,
               error_kw={'elinewidth': 0.8, 'ecolor': (0, 0, 0, 0.35)},
               color=C_IMG, edgecolor='black', linewidth=0.5,
               label='Image Only' if cohort == COHORTS[0] else '')
        ax.bar(x + w/2, sq_vals, w, yerr=sq_yerr, capsize=2,
               error_kw={'elinewidth': 0.8, 'ecolor': (0, 0, 0, 0.35)},
               color=C_SQ, edgecolor='black', linewidth=0.5,
               label='SPARC-Risk' if cohort == COHORTS[0] else '')
    else:
        ax.bar(x - w/2, img_vals, w, color=C_IMG, edgecolor='black', linewidth=0.5,
               label='Image Only' if cohort == COHORTS[0] else '')
        ax.bar(x + w/2, sq_vals, w, color=C_SQ, edgecolor='black', linewidth=0.5,
               label='SPARC-Risk' if cohort == COHORTS[0] else '')

    # Delta annotations (rounded to 2 decimals)
    for i, ep in enumerate(endpoints):
        d = sq_vals[i] - img_vals[i]
        y_top = max(sq_vals[i], img_vals[i])
        if has_ci:
            y_top += max(sq_errs[i][1], img_errs[i][1])
        y_top += 0.008
        color = '#2B6B8A' if d > 0 else '#999999'
        d_rounded = round(d, 2)
        ax.text(i, y_top, f'{d_rounded:+.2f}', ha='center', va='bottom', fontsize=8,
                color=color, fontweight='bold' if abs(d_rounded) >= 0.01 else 'normal')

    ax.set_xticks(x)
    ax.set_xticklabels(endpoints, fontsize=10)
    ax.set_title(f'{cohort}\n(n={r["n"]})', fontsize=11)
    ax.set_ylim(0.48, 0.85)
    ax.grid(axis='y', alpha=0.2)
    ax.axhline(y=0.5, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)

axes[0].set_ylabel('C-index', fontsize=12)
fig.legend(['Image Only', 'SPARC-Risk'], loc='upper right',
           bbox_to_anchor=(0.98, 0.98), fontsize=10)
fig.suptitle('Finetuned External Validation (Ridge Cox)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figs/external_finetuned.png', dpi=300, bbox_inches='tight')
plt.show()